Phase 2b — Duplicate feature vectors and conflicting labels (features_v3)
The audit showed that, after feature engineering, many records share an identical feature vector
(Config 1: 43% of test rows have a vector seen in training; 20.8% of training rows share a vector
with a different label). For every configuration:
  1. the three P1 splits are pooled (they come from the same cleaned dataset), and records with a negative
     time value (duration or inter-arrival time) or a negative header length are removed as invalid values,
     in the same way as missing and infinite values (Misrak & Melaku, 2025); Init_Win_bytes = -1 is kept,
     since it encodes that no initial window was observed rather than an error;
  2. records with an identical feature vector are collapsed into one record;
  3. its label is the majority label of the group (Bayes classifier: Hastie et al., 2009;
     majority vote over repeated labels: Sheng et al., 2008); groups with a tie have no majority
     and are removed (removal of unresolvable inconsistent labels: Croft et al., 2023);
  4. a new stratified 64/16/20 split is made (Misrak & Melaku, 2025), seed 42;
  5. feature engineering (zero-variance / identical columns, log(1+x), Min-Max) is refitted on the
     new training split only, exactly as in P2;
  6. the same rule is applied once more to the scaled float32 vectors (scaling can make distinct raw values
     identical), and validation/test records whose vector occurs in an earlier split are removed.
Outputs go to outputs_v3/features_v3/ (features_v2 is kept unchanged). CPU only.

In [ ]:
# %% ---- cell 1
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% ---- cell 2
import os, json, time, gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT  = '/content/drive/MyDrive/Thesis/outputs_v3/'
DATA  = ROOT + 'data/'
FEAT  = ROOT + 'features_v3/'
SEED  = 42
CLASSES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
WIDE_RANGE = 1_000
SPLITS = ['train', 'val', 'test']

def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)

def pool(ds):
    """The three P1 splits of a dataset, without records holding invalid negative values (step 1)."""
    df = pd.concat([pd.read_parquet(DATA + f'{ds}_{s}.parquet') for s in SPLITS], ignore_index=True)
    cols = [c for c in df.columns if c not in ('label', 'label_orig')
            and any(k in c.lower() for k in ('duration', 'iat', 'header'))]
    bad = (df[cols] < 0).any(axis=1).to_numpy()
    INVALID[ds] = dict(records=int(len(df)), removed_negative_time_or_header=int(bad.sum()),
                       by_class=df.loc[bad, 'label'].astype(str).value_counts().to_dict(),
                       columns_checked=cols)
    log(f'{ds}: removed {int(bad.sum()):,} records with a negative time or header-length value')
    return df[~bad].reset_index(drop=True)

INVALID = {}

def vec_hash(X):
    """Hash of each row; + 0 turns -0.0 into 0.0 so both hash alike."""
    return pd.util.hash_pandas_object(pd.DataFrame(np.asarray(X, np.float32) + np.float32(0)), index=False).to_numpy()

def majority_keep(h, lab):
    """Indices of one record per identical vector, carrying the majority label; vectors with a tie are dropped."""
    cnt = pd.DataFrame({'h': h, 'y': lab}).value_counts().rename('n').reset_index()
    top = cnt.groupby('h')['n'].transform('max')
    best = cnt[cnt['n'] == top]
    nbest = best.groupby('h')['y'].transform('size')
    tie_h = best.loc[nbest > 1, 'h'].unique()
    maj = best[nbest == 1].set_index('h')['y']
    nlab = cnt.groupby('h')['y'].size()
    conflict_h = nlab[nlab > 1].index
    k = pd.DataFrame({'h': h, 'y': lab, 'i': np.arange(len(h))})
    k = k[k['h'].map(maj).eq(k['y'])]
    keep = np.sort(k.drop_duplicates('h')['i'].to_numpy())
    rep = dict(records_in=int(len(h)), unique_vectors=int(len(nlab)),
               duplicate_records_removed=int(len(h) - len(nlab)),
               vectors_with_conflicting_labels=int(len(conflict_h)),
               records_in_conflicting_vectors=int(np.isin(h, conflict_h).sum()),
               vectors_resolved_by_majority=int(len(conflict_h) - len(tie_h)),
               tie_vectors_removed=int(len(tie_h)), records_out=int(len(keep)))
    return keep, rep

def collapse(df, feats, extra=()):
    """Step 2-3 on the raw feature values: one record per identical vector, majority label, ties removed."""
    keep, rep = majority_keep(vec_hash(df[feats].to_numpy(np.float32)), df['label'].astype(str).to_numpy())
    return df.iloc[keep][list(feats) + ['label'] + list(extra)].reset_index(drop=True), rep

def clean_scaled(X, frames):
    """After scaling, distinct raw values can become identical (float32 precision after log/Min-Max).
    The same rule is applied again to the vectors the model actually sees: within each split one record per
    vector (majority label, ties removed), and validation/test records whose vector occurs in an earlier split
    (train; train or validation) are removed, so that no evaluated vector was seen during training."""
    rep, seen = {}, None
    for s in SPLITS:
        h = vec_hash(X[s]); lab = frames[s]['label'].astype(str).to_numpy()
        n0 = len(h); removed_seen = 0
        if seen is not None:
            m = ~np.isin(h, seen); removed_seen = int((~m).sum())
            X[s], frames[s], h, lab = X[s][m], frames[s][m].reset_index(drop=True), h[m], lab[m]
        keep, r = majority_keep(h, lab)
        X[s], frames[s] = X[s][keep], frames[s].iloc[keep].reset_index(drop=True)
        r['removed_seen_in_earlier_split'] = removed_seen; r['records_in'] = n0
        rep[s] = r
        hs = h[keep]; seen = hs if seen is None else np.concatenate([seen, hs])
    return X, frames, rep

def split(df, strat):
    idx = np.arange(len(df))
    trv, te = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=strat)
    tr, va = train_test_split(trv, test_size=0.20, random_state=SEED, stratify=strat[trv])
    return {s: df.iloc[np.sort(ix)].reset_index(drop=True) for s, ix in [('train', tr), ('val', va), ('test', te)]}

def drop_redundant(train, feats):
    const = [c for c in feats if train[c].nunique(dropna=False) <= 1]
    keep, dup = [], {}
    for c in feats:
        if c in const:
            continue
        twin = next((k for k in keep if np.array_equal(train[c].to_numpy(), train[k].to_numpy())), None)
        if twin is None:
            keep.append(c)
        else:
            dup[c] = twin
    return keep, const, dup

def fit_transform(frames, feats):
    tr = frames['train'][feats].to_numpy(np.float64)
    mn, mx = tr.min(0), tr.max(0)
    use_log = ((mx - mn) > WIDE_RANGE) & (mn >= 0)
    lo = np.where(use_log, np.log1p(np.maximum(mn, 0)), mn)
    hi = np.where(use_log, np.log1p(np.maximum(mx, 0)), mx)
    rng = np.where(hi > lo, hi - lo, 1.0)
    out = {}
    for s, df in frames.items():
        x = np.array(df[feats].to_numpy(np.float64), copy=True)
        x[:, use_log] = np.log1p(np.maximum(x[:, use_log], 0))
        out[s] = ((x - lo) / rng).astype(np.float32)
    params = dict(features=feats, train_min=mn.tolist(), train_max=mx.tolist(),
                  log_features=[f for f, u in zip(feats, use_log) if u], wide_range_threshold=WIDE_RANGE)
    return out, params

def overlap_check(X):
    ht = vec_hash(X['train'])
    return {s: round(100 * float(np.isin(vec_hash(X[s]), ht).mean()), 4) for s in ['val', 'test']}

def save(cfg, X, frames, params, extra):
    d = FEAT + cfg + '/'
    os.makedirs(d, exist_ok=True)
    for s in SPLITS:
        np.save(d + f'X_{s}.npy', X[s])
        np.save(d + f'y_{s}.npy', frames[s]['label'].astype(str).map(CLASSES.index).to_numpy(np.int8))
        if 'source' in frames[s]:
            np.save(d + f'source_{s}.npy', frames[s]['source'].to_numpy().astype('U12'))
    json.dump(dict(params, classes=CLASSES, **extra), open(d + 'spec.json', 'w'), indent=2)
    log(f'{cfg}: saved ' + ', '.join(f'{s} {X[s].shape}' for s in SPLITS))

def class_counts(frames):
    return {s: frames[s]['label'].astype(str).value_counts().reindex(CLASSES, fill_value=0).astype(int).to_dict() for s in SPLITS}

In [ ]:
# %% ---- cell 3: Config 2 (CICIDS2017) and Config 3 (CIC-IoT2023)
# earlier runs of this notebook are kept under a new name instead of being overwritten
for c in ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']:
    p = FEAT + c + '/spec.json'
    if os.path.exists(p):
        sp = json.load(open(p))
        tag = '_without_step6' if 'dedup_after_scaling' not in sp else ('_without_invalid_removal' if 'invalid_removed' not in sp else None)
        if tag and not os.path.exists(FEAT + c + tag):
            os.rename(FEAT + c, FEAT + c + tag); log(f'{c}: earlier output renamed to {c}{tag}')
for cfg, ds in [('config2_cicids2017', 'cicids2017'), ('config3_ciciot2023', 'ciciot2023')]:
    if os.path.exists(FEAT + cfg + '/spec.json'):
        log(f'{cfg}: already built, skipping'); continue
    df = pool(ds)
    feats_all = [c for c in df.columns if c not in ('label', 'label_orig')]
    log(f'{cfg}: pooled {len(df):,} records, {len(feats_all)} features')
    df, rep = collapse(df, feats_all)
    log(f'{cfg}: ' + json.dumps(rep))
    frames = split(df, df['label'].astype(str).to_numpy()); del df; gc.collect()
    feats, const, dup = drop_redundant(frames['train'], feats_all)
    X, params = fit_transform(frames, feats)
    ov_before = overlap_check(X)
    X, frames, rep_scaled = clean_scaled(X, frames)
    ov = overlap_check(X)
    log(f'{cfg}: {len(feats)} features | after scaling {json.dumps(rep_scaled)} | % seen in train before/after: {ov_before} / {ov}')
    save(cfg, X, frames, params, dict(invalid_removed=INVALID[ds], dedup_majority=rep, dedup_after_scaling=rep_scaled, overlap_before_final=ov_before, overlap_after=ov, class_counts=class_counts(frames),
                                      dropped_zero_variance=const, dropped_duplicates=dup,
                                      n_features_original=len(feats_all)))
    del frames, X; gc.collect()

In [ ]:
# %% ---- cell 4: Config 1 (merged): 13 unified features
SAME = {
    'fin_flag': ('FIN Flag Count', 'fin_flag_number'),
    'rst_flag': ('RST Flag Count', 'rst_flag_number'), 'psh_flag': ('PSH Flag Count', 'psh_flag_number'),
    'ack_flag': ('ACK Flag Count', 'ack_flag_number'), 'ece_flag': ('ECE Flag Count', 'ece_flag_number'),
    'min_pkt_len': ('Min Packet Length', 'Min'), 'max_pkt_len': ('Max Packet Length', 'Max'),
    'mean_pkt_len': ('Packet Length Mean', 'AVG'), 'pkt_variance': ('Packet Length Variance', 'Variance'),
    'pkt_std': ('Packet Length Std', 'Std'),
}
UNIFIED = list(SAME) + ['iat_mean', 'total_size', 'header_length']

def unify_ids(df):
    u = pd.DataFrame({k: df[v[0]] for k, v in SAME.items()})
    u['iat_mean'] = df['Flow IAT Mean'].clip(lower=0)
    u['total_size'] = df['Total Length of Fwd Packets'] + df['Total Length of Bwd Packets']
    u['header_length'] = (df['Fwd Header Length'].clip(lower=0) / df['Total Fwd Packets'].clip(lower=1)).clip(upper=60)
    u['label'], u['source'] = df['label'].astype(str).values, 'CICIDS2017'
    return u

def unify_iot(df):
    u = pd.DataFrame({k: df[v[1]] for k, v in SAME.items()})
    u['iat_mean'] = df['IAT'].clip(lower=0) * 1e6
    u['total_size'] = df['Tot sum']
    u['header_length'] = df['Header_Length']
    u['label'], u['source'] = df['label'].astype(str).values, 'CIC-IoT2023'
    return u

cfg = 'config1_merged'
if not os.path.exists(FEAT + cfg + '/spec.json'):
    df = pd.concat([unify_ids(pool('cicids2017')), unify_iot(pool('ciciot2023'))], ignore_index=True)
    log(f'{cfg}: pooled {len(df):,} records')
    # how many identical vectors come from both datasets (reported; the record kept takes its source)
    hh = vec_hash(df[UNIFIED].to_numpy(np.float32))
    cross = int((pd.DataFrame({'h': hh, 's': df['source'].values}).groupby('h')['s'].nunique() > 1).sum()); del hh
    df, rep = collapse(df, UNIFIED, extra=['source'])
    rep['vectors_present_in_both_datasets'] = cross
    log(f'{cfg}: ' + json.dumps(rep))
    strat = (df['label'].astype(str) + '|' + df['source'].astype(str)).to_numpy()   # keeps each dataset's share in every split
    frames = split(df, strat); del df; gc.collect()
    frames = {s: f.sample(frac=1, random_state=SEED).reset_index(drop=True) for s, f in frames.items()}
    X, params = fit_transform(frames, UNIFIED)
    ov_before = overlap_check(X)
    X, frames, rep_scaled = clean_scaled(X, frames)
    ov = overlap_check(X)
    log(f'{cfg}: after scaling {json.dumps(rep_scaled)} | % seen in train before/after: {ov_before} / {ov}')
    save(cfg, X, frames, params, dict(invalid_removed={k: INVALID[k] for k in ('cicids2017', 'ciciot2023')}, dedup_majority=rep, dedup_after_scaling=rep_scaled, overlap_before_final=ov_before, overlap_after=ov, class_counts=class_counts(frames),
                                      source_counts={s: frames[s]['source'].value_counts().to_dict() for s in SPLITS},
                                      mapping={k: list(v) for k, v in SAME.items()}))

In [ ]:
# %% ---- cell 5: summary
for c in ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']:
    sp = json.load(open(FEAT + c + '/spec.json'))
    print(c, 'invalid removed', json.dumps(sp['invalid_removed']), '|', json.dumps(sp['dedup_majority']), '| after scaling', json.dumps(sp['dedup_after_scaling']), '| overlap', sp['overlap_after'], '| features', len(sp['features']))
    print('   train classes', sp['class_counts']['train'])